In [ ]:
"""
シャドーバース風カードゲーム 戦闘システム
改訂版
"""

import random
import copy

# 効果定数
EFCT_NONE         = 0  # 効果なし
EFCT_RUSH         = 1  # 疾走（相手フォロワーにのみ攻撃可・召喚酔いなし）
EFCT_CHARGE       = 2  # 突進（相手プレイヤーにも攻撃可・召喚酔いなし）
EFCT_WARD         = 3  # ガード（守護）
EFCT_WARD_CHARGE  = 4  # 守護＋突進

SPELL_DAMAGE = 1
SPELL_DRAW   = 2

MAX_HAND      = 10
MAX_STAGE     = 5
INITIAL_HAND  = 4
PLAYER_HP     = 20
MAX_PP        = 10

# カードデータ
CARD_MASTER = [
    {"name": "ダメージ魔法", "cost": 1, "hp": 0, "atk": 3, "efct": EFCT_NONE,        "spell": SPELL_DAMAGE, "order": 0},
    {"name": "ドロー魔法",   "cost": 1, "hp": 0, "atk": 0, "efct": EFCT_NONE,        "spell": SPELL_DRAW,   "order": 0},
    {"name": "トラ",         "cost": 1, "hp": 1, "atk": 1, "efct": EFCT_CHARGE,      "spell": 0, "order": 0},
    {"name": "キジ",         "cost": 2, "hp": 3, "atk": 1, "efct": EFCT_RUSH,        "spell": 0, "order": 0},
    {"name": "サル",         "cost": 2, "hp": 2, "atk": 3, "efct": EFCT_RUSH,        "spell": 0, "order": 0},
    {"name": "ブタ",         "cost": 3, "hp": 1, "atk": 5, "efct": EFCT_CHARGE,      "spell": 0, "order": 0},
    {"name": "リス",         "cost": 3, "hp": 5, "atk": 3, "efct": EFCT_WARD,        "spell": 0, "order": 1},
    {"name": "カバ",         "cost": 4, "hp": 7, "atk": 4, "efct": EFCT_WARD_CHARGE, "spell": 0, "order": 1},
    {"name": "ウマ",         "cost": 5, "hp": 5, "atk": 5, "efct": EFCT_NONE,        "spell": 0, "order": 0},
    {"name": "クマ",         "cost": 6, "hp": 9, "atk": 9, "efct": EFCT_NONE,        "spell": 0, "order": 0},
]

class GameState:
    def __init__(self):
        self.hp    = {"A": PLAYER_HP, "B": PLAYER_HP}
        self.hand  = {"A": [], "B": []}
        self.stage = {"A": [], "B": []}
        self.deck  = {
            "A": [copy.deepcopy(c) for c in CARD_MASTER],
            "B": [copy.deepcopy(c) for c in CARD_MASTER],
        }
        self.evo   = {"A": 2, "B": 2}
        self.turn  = 0
        self.pp    = 0
        self.first = ""

    def current_player(self):
        if self.first == "A":
            return "A" if self.turn % 2 == 1 else "B"
        else:
            return "B" if self.turn % 2 == 1 else "A"

    def opponent(self):
        cp = self.current_player()
        return "B" if cp == "A" else "A"


def safe_int(prompt, lo, hi):
    while True:
        try:
            val = int(input(prompt))
            if lo <= val <= hi:
                return val
            print(f"  ※ {lo}〜{hi} の数字を入力してください")
        except ValueError:
            print("  ※ 数字を入力してください")


def fmt_card(card):
    efct_label = {
        EFCT_NONE:        "",
        EFCT_RUSH:        "[疾走]",
        EFCT_CHARGE:      "[突進]",
        EFCT_WARD:        "[守護]",
        EFCT_WARD_CHARGE: "[守護+突進]",
    }.get(card["efct"], "")
    evo_mark = "★" if card.get("evo") else ""
    return f"{card['name']}{evo_mark}{efct_label}(ATK:{card['atk']} HP:{card['hp']} cost:{card['cost']})"


def show_stage(gs):
    cp  = gs.current_player()
    opp = gs.opponent()
    print(f"\n{'='*50}")
    print(f"  ── {cp}さんのターン（ターン {gs.turn} / PP {gs.pp}）──")
    print(f"  あなたのHP: {gs.hp[cp]}  相手のHP: {gs.hp[opp]}")
    print(f"  残り進化回数: {gs.evo[cp]}")
    print()
    print(f"  【相手の盤面 ({opp})】")
    if gs.stage[opp]:
        for i, c in enumerate(gs.stage[opp], 1):
            print(f"    {i}: {fmt_card(c)}")
    else:
        print("    （なし）")
    print()
    print(f"  【自分の盤面 ({cp})】")
    if gs.stage[cp]:
        for i, c in enumerate(gs.stage[cp], 1):
            atk_ok = "○" if c.get("turn_count", 0) > 0 else "×"
            print(f"    {i}: {fmt_card(c)}  攻撃:{atk_ok}")
    else:
        print("    （なし）")
    print()
    print(f"  【手札 ({cp})】")
    for i, c in enumerate(gs.hand[cp], 1):
        print(f"    {i}: {fmt_card(c)}")
    print(f"{'='*50}")

def turn_zero(gs):
    gs.first = random.choice(["A", "B"])
    print(f"\n【コイントス】{gs.first}さんが先攻です！")
    random.shuffle(gs.deck["A"])
    random.shuffle(gs.deck["B"])


def draw_initial_hand(gs, player):
    gs.hand[player] = [gs.deck[player].pop(0) for _ in range(INITIAL_HAND)]
    print(f"\n【{player}さんの初期手札】")
    for i, c in enumerate(gs.hand[player], 1):
        print(f"  {i}: {fmt_card(c)}")
    count = safe_int(f"{player}さん、引き直す枚数（0〜{INITIAL_HAND}）: ", 0, INITIAL_HAND)
    if count == 0:
        return
    indices = []
    for _ in range(count):
        idx = safe_int(f"  引き直すカードの番号（1〜{len(gs.hand[player])}）: ", 1, len(gs.hand[player])) - 1
        if idx not in indices:
            indices.append(idx)
    for i in sorted(indices, reverse=True):
        gs.deck[player].append(gs.hand[player].pop(i))
    random.shuffle(gs.deck[player])
    for _ in range(len(indices)):
        if gs.deck[player]:
            gs.hand[player].append(gs.deck[player].pop(0))
    print(f"  【{player}さんの新しい手札】")
    for i, c in enumerate(gs.hand[player], 1):
        print(f"    {i}: {fmt_card(c)}")


def start_turn(gs):
    gs.turn += 1
    cp  = gs.current_player()
    opp = gs.opponent()
    gs.pp = min((gs.turn + 1) // 2, MAX_PP)
    for card in gs.stage[cp]:
        card["turn_count"] = card.get("turn_count", 0) + 1
    for card in gs.stage[opp]:
        card["turn_count"] = card.get("turn_count", 0) + 1
    if gs.deck[cp]:
        drawn = gs.deck[cp].pop(0)
        if len(gs.hand[cp]) < MAX_HAND:
            gs.hand[cp].append(drawn)
            print(f"\n  {cp}さんが「{drawn['name']}」をドローしました")
        else:
            print(f"\n  {cp}さんの手札が上限のため「{drawn['name']}」は捨てられました")
    else:
        print(f"\n  {cp}さんのデッキが空です！")


def apply_spell(gs, card, caster):
    opp = "B" if caster == "A" else "A"
    if card["spell"] == SPELL_DAMAGE:
        for c in gs.stage[opp]:
            c["hp"] -= card["atk"]
        gs.stage[opp] = [c for c in gs.stage[opp] if c["hp"] > 0]
        print(f"  ✦ ダメージスペル発動！相手フォロワー全体に {card['atk']} ダメージ")
    elif card["spell"] == SPELL_DRAW:
        drawn = 0
        for _ in range(2):
            if gs.deck[caster] and len(gs.hand[caster]) < MAX_HAND:
                gs.hand[caster].append(gs.deck[caster].pop(0))
                drawn += 1
        print(f"  ✦ ドロースペル発動！{drawn} 枚ドローしました")


def play_card(gs):
    cp = gs.current_player()
    while True:
        show_stage(gs)
        print(f"  PP残り: {gs.pp}")
        choice = safe_int("  プレイするカード番号（0で終了）: ", 0, len(gs.hand[cp]))
        if choice == 0:
            break
        card = gs.hand[cp][choice - 1]
        if gs.pp < card["cost"]:
            print("  ※ PPが足りません")
            continue
        gs.pp -= card["cost"]
        if card["spell"] > 0:
            apply_spell(gs, card, cp)
            gs.hand[cp].pop(choice - 1)
        else:
            if len(gs.stage[cp]) >= MAX_STAGE:
                print(f"  ※ 盤面が満員（上限 {MAX_STAGE} 体）のため出せません")
                gs.pp += card["cost"]
                continue
            new_card = copy.deepcopy(card)
            new_card["evo"] = 0
            new_card["turn_count"] = 0
            gs.stage[cp].append(new_card)
            gs.hand[cp].pop(choice - 1)
            print(f"  ✦ 「{new_card['name']}」を召喚！")
            if new_card["efct"] in (EFCT_CHARGE, EFCT_WARD_CHARGE):
                new_card["turn_count"] = 1
                print("    （突進効果：このターンから攻撃可能）")
            elif new_card["efct"] == EFCT_RUSH:
                new_card["turn_count"] = 1
                print("    （疾走効果：このターンからフォロワーに攻撃可能）")


def evolution(gs):
    cp = gs.current_player()
    if not gs.stage[cp] or gs.evo[cp] <= 0:
        return
    ans = input("  進化しますか？ (Y/N): ").strip().upper()
    if ans != "Y":
        return
    for i, c in enumerate(gs.stage[cp], 1):
        print(f"    {i}: {fmt_card(c)} {'（進化済み）' if c.get('evo') else ''}")
    idx = safe_int(f"  進化するフォロワー（1〜{len(gs.stage[cp])}）: ", 1, len(gs.stage[cp])) - 1
    target = gs.stage[cp][idx]
    if target.get("evo"):
        print("  ※ すでに進化済みです")
        return
    target["evo"] = 1
    target["hp"]  += 2
    target["atk"] += 2
    target["turn_count"] = max(target.get("turn_count", 0), 1)
    gs.evo[cp] -= 1
    print(f"  ★ 「{target['name']}」が進化！(ATK+2 HP+2) 残り進化回数: {gs.evo[cp]}")

def has_ward(stage):
    return any(c["efct"] in (EFCT_WARD, EFCT_WARD_CHARGE) for c in stage)


def get_valid_attackers(gs, cp):
    return [i for i, c in enumerate(gs.stage[cp]) if c.get("turn_count", 0) > 0]


def get_valid_targets(gs, cp, attacker):
    opp = "B" if cp == "A" else "A"
    targets = []
    if has_ward(gs.stage[opp]):
        for i, c in enumerate(gs.stage[opp]):
            if c["efct"] in (EFCT_WARD, EFCT_WARD_CHARGE):
                targets.append((f"フォロワー{i+1}:{fmt_card(c)}", "follower", i))
        return targets
    if attacker["efct"] == EFCT_RUSH:
        for i, c in enumerate(gs.stage[opp]):
            targets.append((f"フォロワー{i+1}:{fmt_card(c)}", "follower", i))
        return targets
    targets.append((f"プレイヤー({opp}) HP:{gs.hp[opp]}", "player", -1))
    for i, c in enumerate(gs.stage[opp]):
        targets.append((f"フォロワー{i+1}:{fmt_card(c)}", "follower", i))
    return targets


def do_attack(gs, cp, attacker_idx, target_kind, target_idx):
    opp = "B" if cp == "A" else "A"
    atk = gs.stage[cp][attacker_idx]
    if target_kind == "player":
        gs.hp[opp] -= atk["atk"]
        print(f"  ⚔ 「{atk['name']}」が{opp}プレイヤーに {atk['atk']} ダメージ！（残りHP: {gs.hp[opp]}）")
    else:
        dfn = gs.stage[opp][target_idx]
        print(f"  ⚔ 「{atk['name']}」vs「{dfn['name']}」")
        atk["hp"] -= dfn["atk"]
        dfn["hp"] -= atk["atk"]
        atk_destroyed = atk["hp"] <= 0
        dfn_destroyed = dfn["hp"] <= 0
        if dfn_destroyed:
            print(f"    「{dfn['name']}」は破壊された！")
        else:
            print(f"    「{dfn['name']}」残りHP: {dfn['hp']}")
        if atk_destroyed:
            print(f"    「{atk['name']}」も破壊された！")
        else:
            print(f"    「{atk['name']}」残りHP: {atk['hp']}")
        if dfn_destroyed:
            gs.stage[opp].pop(target_idx)
        if atk_destroyed:
            gs.stage[cp].pop(attacker_idx)
            return
    if attacker_idx < len(gs.stage[cp]):
        gs.stage[cp][attacker_idx]["turn_count"] = 0


def battle(gs):
    cp = gs.current_player()
    if not gs.stage[cp]:
        print("  盤面にフォロワーがいません")
        return
    while True:
        attackers = get_valid_attackers(gs, cp)
        if not attackers:
            print("  攻撃可能なフォロワーがいません")
            break
        show_stage(gs)
        for i in attackers:
            print(f"    {i+1}: {fmt_card(gs.stage[cp][i])}")
        at_choice = safe_int("  攻撃するフォロワー番号（0で終了）: ", 0, len(gs.stage[cp]))
        if at_choice == 0:
            break
        at_idx = at_choice - 1
        if at_idx not in attackers:
            print("  ※ そのフォロワーは攻撃できません")
            continue
        attacker = gs.stage[cp][at_idx]
        targets = get_valid_targets(gs, cp, attacker)
        if not targets:
            print("  攻撃対象がありません")
            continue
        for i, (label, _, _) in enumerate(targets, 1):
            print(f"    {i}: {label}")
        tgt_choice = safe_int("  攻撃対象の番号: ", 1, len(targets)) - 1
        _, kind, idx = targets[tgt_choice]
        do_attack(gs, cp, at_idx, kind, idx)


def check_result(gs):
    results = []
    if gs.hp["A"] <= 0:
        results.append("Bさんの勝ちです！（AのHPが0）")
    if gs.hp["B"] <= 0:
        results.append("Aさんの勝ちです！（BのHPが0）")
    if not gs.deck["A"] and not gs.hand["A"] and not gs.stage["A"]:
        results.append("Bさんの勝ちです！（Aのカードが尽きた）")
    if not gs.deck["B"] and not gs.hand["B"] and not gs.stage["B"]:
        results.append("Aさんの勝ちです！（Bのカードが尽きた）")
    if results:
        print("\n" + "="*50)
        for r in results:
            print(f"  🏆 {r}")
        print("="*50)
        return False
    return True


def main():
    print("\n" + "="*50)
    print("  シャドーバース風カードゲーム")
    print("="*50)
    gs = GameState()
    turn_zero(gs)
    draw_initial_hand(gs, "A")
    draw_initial_hand(gs, "B")
    while check_result(gs):
        start_turn(gs)
        cp = gs.current_player()
        print(f"\n  ▶ {cp}さんのターン開始！（PP: {gs.pp}）")
        play_card(gs)
        if not check_result(gs):
            break
        evolution(gs)
        battle(gs)
        if not check_result(gs):
            break
        input("\n  Enterキーでターンを終了します...")
    print("\nゲーム終了。ありがとうございました！\n")


main()